In [10]:
import os
import requests
from openai import OpenAI

client = OpenAI()

# --- 1. 실제 영화 데이터를 가져오는 API 함수들 ---
BASE_URL = "https://nomad-movies-2.nomadcoders.workers.dev"

def get_popular_movies():
    response = requests.get(f"{BASE_URL}/movies")
    return response.json()

def get_movie_details(id):
    response = requests.get(f"{BASE_URL}/movies/{id}")
    return response.json()

def get_movie_credits(id):
    response = requests.get(f"{BASE_URL}/movies/{id}/credits")
    return response.json()

# --- 2. LLM에게 어떤 함수를 쓸지 물어보는 함수 ---
def get_movie_function(user_question):
    PROMPT = f"""
    I have the following functions in my system.
    
    'get_popular_movies()' - 인기 영화 목록을 가져옵니다.
    'get_movie_details(id)' - ID로 영화 상세 정보를 조회합니다.
    'get_movie_credits(id)' - 영화의 출연진 및 제작진을 조회합니다.
    
    All of them receive the movie id as an argument if needed (i.e get_movie_details(550))
    
    Please answer with the name of the function that you would like me to run.
    Please say nothing else, just the name of the function with the arguments.
    
    Answer the following question:
    {user_question}
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": PROMPT
            }
        ]
    )
    
    return response.choices[0].message.content

In [11]:
test_questions = [
    "지금 인기 있는 영화가 무엇인지 알려줘",
    "movie ID 550에 해당하는 영화가 무엇인지 알려줘",
    "movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘"
]

for question in test_questions:
    print(f"🙋‍♂️ 질문: {question}")
    
    # 1. LLM에게 물어보기
    func_string = get_movie_function(question)
    print(f"🤖 LLM의 선택: {func_string}")
    
    # 2. LLM이 고른 함수를 실제로 실행해서 데이터 가져오기 (eval 함수 사용)
    try:
        print("⏳ 영화 데이터 가져오는 중...")
        
        # eval()은 문자열 코드를 실제 파이썬 코드로 실행해주는 명령어입니다.
        result = eval(func_string) 
        
        # 데이터가 너무 길게 나오니, 앞부분 200글자만 잘라서 출력해봅니다.
        print(f"🎬 결과 확인: {str(result)[:200]}...") 
        
    except Exception as e:
        print(f"❌ 에러 발생: {e}")
        
    print("-" * 60)

🙋‍♂️ 질문: 지금 인기 있는 영화가 무엇인지 알려줘
🤖 LLM의 선택: get_popular_movies()
⏳ 영화 데이터 가져오는 중...
🎬 결과 확인: [{'adult': False, 'backdrop_path': 'https://image.tmdb.org/t/p/w1280/4k99kV4R1bbbrsnjR205v91Xbin.jpg', 'genre_ids': [27], 'id': 1339713, 'title': 'Obsession', 'original_language': 'en', 'original_titl...
------------------------------------------------------------
🙋‍♂️ 질문: movie ID 550에 해당하는 영화가 무엇인지 알려줘
🤖 LLM의 선택: get_movie_details(550)
⏳ 영화 데이터 가져오는 중...
🎬 결과 확인: {'adult': False, 'backdrop_path': 'https://image.tmdb.org/t/p/w1280/xRyINp9KfMLVjRiO5nCsoRDdvvF.jpg', 'belongs_to_collection': None, 'budget': 63000000, 'genres': [{'id': 18, 'name': 'Drama'}, {'id': ...
------------------------------------------------------------
🙋‍♂️ 질문: movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘
🤖 LLM의 선택: get_movie_credits(550)
⏳ 영화 데이터 가져오는 중...
🎬 결과 확인: [{'adult': False, 'gender': 2, 'id': 819, 'known_for_department': 'Acting', 'name': 'Edward Norton', 'original_name': 'Edward Norton', 'popularity': 4.6098, 'profile_pat